In [66]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from tqdm.auto import tqdm
import re

## Comments Dataset

In [67]:
comments = pd.read_csv(r'C:\Users\New Owner\OneDrive\Documents\April DS Code Jam\youtube_sentiment_analysis\datasets\archive\UScomments.csv', on_bad_lines='skip', encoding= 'utf-8', low_memory=False) # Load the dataset
display(comments.head()) # Display the first few rows of the dataset
display(comments.info()) # Display the information about the dataset

,video_id,comment_text,likes,replies
0,XpVt6Z1Gjjo,Logan Paul it's yo big day ‼️‼️‼️,4,0
1,XpVt6Z1Gjjo,I've been following you from the start of your...,3,0
2,XpVt6Z1Gjjo,Say hi to Kong and maverick for me,3,0
3,XpVt6Z1Gjjo,MY FAN . attendance,3,0
4,XpVt6Z1Gjjo,trending 😉,3,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 691400 entries, 0 to 691399
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   video_id      691400 non-null  object
 1   comment_text  691374 non-null  object
 2   likes         691400 non-null  object
 3   replies       691400 non-null  object
dtypes: object(4)
memory usage: 21.1+ MB


None

In [68]:
comments.loc[comments['comment_text'].isna()]  # Filter rows based on condition

,video_id,comment_text,likes,replies
1417,1L7JFN7tQLs,NaN,0,0
76134,7YAAyUFL1GQ,NaN,0,0
215218,KUCHBBCj77I,NaN,0,0
234226,KUCHBBCj77I,NaN,0,0
306019,s3Hk_lDw5yo,NaN,0,0
332811,zrOHeEA14kQ,NaN,0,0
357506,zmg9tVaMVd4,NaN,0,0
379582,zmg9tVaMVd4,NaN,0,0
403013,9eea7_7OBZQ,NaN,0,0
425238,6l5P7jHUcjI,NaN,0,0


In [69]:
comments.dropna(inplace=True) # Drop rows with null values

In [70]:
comments.info() # Check the data types and null values

<class 'pandas.core.frame.DataFrame'>
Index: 691374 entries, 0 to 691399
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   video_id      691374 non-null  object
 1   comment_text  691374 non-null  object
 2   likes         691374 non-null  object
 3   replies       691374 non-null  object
dtypes: object(4)
memory usage: 26.4+ MB


## Videos Dataset

In [71]:
videos = pd.read_csv(r'C:\Users\New Owner\OneDrive\Documents\April DS Code Jam\youtube_sentiment_analysis\datasets\archive\USvideos.csv', on_bad_lines='skip', encoding= 'utf-8').head(10) # Display the first 10 rows of the dataset
display(videos) # Display the videos dataset
display(videos.info()) # Display the data types and null values of the videos dataset

,video_id,title,channel_title,category_id,tags,views,likes,dislikes,comment_total,thumbnail_link,date
0,XpVt6Z1Gjjo,1 YEAR OF VLOGGING -- HOW LOGAN PAUL CHANGED Y...,Logan Paul Vlogs,24,logan paul vlog|logan paul|logan|paul|olympics...,4394029,320053,5931,46245,https://i.ytimg.com/vi/XpVt6Z1Gjjo/default.jpg,13.09
1,K4wEI5zhHB0,iPhone X — Introducing iPhone X — Apple,Apple,28,Apple|iPhone 10|iPhone Ten|iPhone|Portrait Lig...,7860119,185853,26679,0,https://i.ytimg.com/vi/K4wEI5zhHB0/default.jpg,13.09
2,cLdxuaxaQwc,My Response,PewDiePie,22,[none],5845909,576597,39774,170708,https://i.ytimg.com/vi/cLdxuaxaQwc/default.jpg,13.09
3,WYYvHb03Eog,Apple iPhone X first look,The Verge,28,apple iphone x hands on|Apple iPhone X|iPhone ...,2642103,24975,4542,12829,https://i.ytimg.com/vi/WYYvHb03Eog/default.jpg,13.09
4,sjlHnJvXdQs,iPhone X (parody),jacksfilms,23,jacksfilms|parody|parodies|iphone|iphone x|iph...,1168130,96666,568,6666,https://i.ytimg.com/vi/sjlHnJvXdQs/default.jpg,13.09
5,cMKX2tE5Luk,The Disaster Artist | Official Trailer HD | A24,A24,1,a24|a24 films|a24 trailers|independent films|t...,1311445,34507,544,3040,https://i.ytimg.com/vi/cMKX2tE5Luk/default.jpg,13.09
6,8wNr-NQImFg,"The Check In: HUD, Ben Carson and Hurricanes",Late Night with Seth Meyers,23,Late night|Seth Meyers|check in|hud|Ben Carson...,666169,9985,297,1071,https://i.ytimg.com/vi/8wNr-NQImFg/default.jpg,13.09
7,_HTXMhKWqnA,iPhone X Impressions & Hands On!,Marques Brownlee,28,iPhone X|iphone x|iphone 10|iPhone X impressio...,1728614,74062,2180,15297,https://i.ytimg.com/vi/_HTXMhKWqnA/default.jpg,13.09
8,_ANP3HR1jsM,ATTACKED BY A POLICE DOG!!,RomanAtwoodVlogs,22,Roman Atwood|Roman|Atwood|roman atwood vlogs|f...,1338533,69687,678,5643,https://i.ytimg.com/vi/_ANP3HR1jsM/default.jpg,13.09
9,zgLtEob6X-Q,Honest Trailers - The Mummy (2017),Screen Junkies,1,screenjunkies|screen junkies|screenjunkies new...,1056891,29943,878,4046,https://i.ytimg.com/vi/zgLtEob6X-Q/default.jpg,13.09


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   video_id        10 non-null     object 
 1   title           10 non-null     object 
 2   channel_title   10 non-null     object 
 3   category_id     10 non-null     int64  
 4   tags            10 non-null     object 
 5   views           10 non-null     int64  
 6   likes           10 non-null     int64  
 7   dislikes        10 non-null     int64  
 8   comment_total   10 non-null     int64  
 9   thumbnail_link  10 non-null     object 
 10  date            10 non-null     float64
dtypes: float64(1), int64(5), object(5)
memory usage: 1012.0+ bytes


None

Note: understand the relationship between the two datasets,
the comments datasets contain all of the comments for a certain video(marked with id)

## Calculating Like-to-Dislike Ratio

In [72]:
videos['l_d_ratio'] = videos['likes'] / videos['dislikes'] # Calculate the like-dislike ratio
videos = videos.drop(columns=['date']) # Drop the date column
videos.head(10)

,video_id,title,channel_title,category_id,tags,views,likes,dislikes,comment_total,thumbnail_link,l_d_ratio
0,XpVt6Z1Gjjo,1 YEAR OF VLOGGING -- HOW LOGAN PAUL CHANGED Y...,Logan Paul Vlogs,24,logan paul vlog|logan paul|logan|paul|olympics...,4394029,320053,5931,46245,https://i.ytimg.com/vi/XpVt6Z1Gjjo/default.jpg,53.962738
1,K4wEI5zhHB0,iPhone X — Introducing iPhone X — Apple,Apple,28,Apple|iPhone 10|iPhone Ten|iPhone|Portrait Lig...,7860119,185853,26679,0,https://i.ytimg.com/vi/K4wEI5zhHB0/default.jpg,6.966266
2,cLdxuaxaQwc,My Response,PewDiePie,22,[none],5845909,576597,39774,170708,https://i.ytimg.com/vi/cLdxuaxaQwc/default.jpg,14.496832
3,WYYvHb03Eog,Apple iPhone X first look,The Verge,28,apple iphone x hands on|Apple iPhone X|iPhone ...,2642103,24975,4542,12829,https://i.ytimg.com/vi/WYYvHb03Eog/default.jpg,5.498679
4,sjlHnJvXdQs,iPhone X (parody),jacksfilms,23,jacksfilms|parody|parodies|iphone|iphone x|iph...,1168130,96666,568,6666,https://i.ytimg.com/vi/sjlHnJvXdQs/default.jpg,170.186620
5,cMKX2tE5Luk,The Disaster Artist | Official Trailer HD | A24,A24,1,a24|a24 films|a24 trailers|independent films|t...,1311445,34507,544,3040,https://i.ytimg.com/vi/cMKX2tE5Luk/default.jpg,63.431985
6,8wNr-NQImFg,"The Check In: HUD, Ben Carson and Hurricanes",Late Night with Seth Meyers,23,Late night|Seth Meyers|check in|hud|Ben Carson...,666169,9985,297,1071,https://i.ytimg.com/vi/8wNr-NQImFg/default.jpg,33.619529
7,_HTXMhKWqnA,iPhone X Impressions & Hands On!,Marques Brownlee,28,iPhone X|iphone x|iphone 10|iPhone X impressio...,1728614,74062,2180,15297,https://i.ytimg.com/vi/_HTXMhKWqnA/default.jpg,33.973394
8,_ANP3HR1jsM,ATTACKED BY A POLICE DOG!!,RomanAtwoodVlogs,22,Roman Atwood|Roman|Atwood|roman atwood vlogs|f...,1338533,69687,678,5643,https://i.ytimg.com/vi/_ANP3HR1jsM/default.jpg,102.783186
9,zgLtEob6X-Q,Honest Trailers - The Mummy (2017),Screen Junkies,1,screenjunkies|screen junkies|screenjunkies new...,1056891,29943,878,4046,https://i.ytimg.com/vi/zgLtEob6X-Q/default.jpg,34.103645


## Text Cleaning

In [73]:
def clear_text(text):
    text = text.lower() # Convert to lowercase
    pattern = r'[^a-zA-Z\s]' # Regular expression pattern to match special characters and punctuation 
    text = re.sub(pattern, " ", text) # Remove special characters, including punctuation
    return text

In [74]:
comments['comment_text'] = comments['comment_text'].astype(str).apply(clear_text)
comments['comment_text'].head(10)

0                    logan paul it s yo big day       
1    i ve been following you from the start of your...
2                   say hi to kong and maverick for me
3                                  my fan   attendance
4                                           trending  
5                                 on trending ayyeeeee
6                                 the end though      
7                                    trending         
8                          happy one year vlogaversary
9    you and your shit brother may have single hand...
Name: comment_text, dtype: object

## Tokenization/ Stop Word Removal / Lemmetization

In [75]:
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')


stop_words = set(stopwords.words('english')) # Set of English stop words
lemmatizer = WordNetLemmatizer() # Initialize the lemmatizer

comments['tokenized_text'] = comments['comment_text'].fillna("").astype(str).apply(word_tokenize) # Apply tokenization to the 'comment_text' column
comments['tokenized_text'] = comments['tokenized_text'].apply(lambda tokens: [word for word in tokens if word not in stop_words]) # Remove stop words from the tokenized text
comments['tokenized_text'] = comments['tokenized_text'].apply(lambda tokens: [lemmatizer.lemmatize(word) for word in tokens]) # Apply lemmatization to the tokenized text

[nltk_data] Downloading package punkt_tab to C:\Users\New
[nltk_data]     Owner\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\New
[nltk_data]     Owner\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\New
[nltk_data]     Owner\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [76]:
# Join the tokenized words back into a single string and save it back to the column
comments['tokenized_text'] = comments['tokenized_text'].apply(lambda tokens: ' '.join(tokens))
comments = comments.drop(columns=['comment_text']) # Drop the original 'comment_text' column

In [77]:
comments.head(10)

,video_id,likes,replies,tokenized_text
0,XpVt6Z1Gjjo,4,0,logan paul yo big day
1,XpVt6Z1Gjjo,3,0,following start vine channel seen vlogs
2,XpVt6Z1Gjjo,3,0,say hi kong maverick
3,XpVt6Z1Gjjo,3,0,fan attendance
4,XpVt6Z1Gjjo,3,0,trending
5,XpVt6Z1Gjjo,3,0,trending ayyeeeee
6,XpVt6Z1Gjjo,4,0,end though
7,XpVt6Z1Gjjo,3,0,trending
8,XpVt6Z1Gjjo,3,0,happy one year vlogaversary
9,XpVt6Z1Gjjo,0,0,shit brother may single handedly ruined youtub...


In [78]:
videos.head(10)

,video_id,title,channel_title,category_id,tags,views,likes,dislikes,comment_total,thumbnail_link,l_d_ratio
0,XpVt6Z1Gjjo,1 YEAR OF VLOGGING -- HOW LOGAN PAUL CHANGED Y...,Logan Paul Vlogs,24,logan paul vlog|logan paul|logan|paul|olympics...,4394029,320053,5931,46245,https://i.ytimg.com/vi/XpVt6Z1Gjjo/default.jpg,53.962738
1,K4wEI5zhHB0,iPhone X — Introducing iPhone X — Apple,Apple,28,Apple|iPhone 10|iPhone Ten|iPhone|Portrait Lig...,7860119,185853,26679,0,https://i.ytimg.com/vi/K4wEI5zhHB0/default.jpg,6.966266
2,cLdxuaxaQwc,My Response,PewDiePie,22,[none],5845909,576597,39774,170708,https://i.ytimg.com/vi/cLdxuaxaQwc/default.jpg,14.496832
3,WYYvHb03Eog,Apple iPhone X first look,The Verge,28,apple iphone x hands on|Apple iPhone X|iPhone ...,2642103,24975,4542,12829,https://i.ytimg.com/vi/WYYvHb03Eog/default.jpg,5.498679
4,sjlHnJvXdQs,iPhone X (parody),jacksfilms,23,jacksfilms|parody|parodies|iphone|iphone x|iph...,1168130,96666,568,6666,https://i.ytimg.com/vi/sjlHnJvXdQs/default.jpg,170.186620
5,cMKX2tE5Luk,The Disaster Artist | Official Trailer HD | A24,A24,1,a24|a24 films|a24 trailers|independent films|t...,1311445,34507,544,3040,https://i.ytimg.com/vi/cMKX2tE5Luk/default.jpg,63.431985
6,8wNr-NQImFg,"The Check In: HUD, Ben Carson and Hurricanes",Late Night with Seth Meyers,23,Late night|Seth Meyers|check in|hud|Ben Carson...,666169,9985,297,1071,https://i.ytimg.com/vi/8wNr-NQImFg/default.jpg,33.619529
7,_HTXMhKWqnA,iPhone X Impressions & Hands On!,Marques Brownlee,28,iPhone X|iphone x|iphone 10|iPhone X impressio...,1728614,74062,2180,15297,https://i.ytimg.com/vi/_HTXMhKWqnA/default.jpg,33.973394
8,_ANP3HR1jsM,ATTACKED BY A POLICE DOG!!,RomanAtwoodVlogs,22,Roman Atwood|Roman|Atwood|roman atwood vlogs|f...,1338533,69687,678,5643,https://i.ytimg.com/vi/_ANP3HR1jsM/default.jpg,102.783186
9,zgLtEob6X-Q,Honest Trailers - The Mummy (2017),Screen Junkies,1,screenjunkies|screen junkies|screenjunkies new...,1056891,29943,878,4046,https://i.ytimg.com/vi/zgLtEob6X-Q/default.jpg,34.103645


## Creating Positive to Negative Comments Ratio

In [79]:
# 1.Calculate the polarity score of an individual comment
from textblob import TextBlob

# Function to calculate sentiment polarity
def get_sentiment(text):
    analysis = TextBlob(text)
    return analysis.sentiment.polarity

# Apply the function to the 'tokenized_text' column
comments['polarity']= comments['tokenized_text'].apply(get_sentiment)

comments.head(10)

,video_id,likes,replies,tokenized_text,polarity
0,XpVt6Z1Gjjo,4,0,logan paul yo big day,0.00000
1,XpVt6Z1Gjjo,3,0,following start vine channel seen vlogs,0.00000
2,XpVt6Z1Gjjo,3,0,say hi kong maverick,0.00000
3,XpVt6Z1Gjjo,3,0,fan attendance,0.00000
4,XpVt6Z1Gjjo,3,0,trending,0.00000
5,XpVt6Z1Gjjo,3,0,trending ayyeeeee,0.00000
6,XpVt6Z1Gjjo,4,0,end though,0.00000
7,XpVt6Z1Gjjo,3,0,trending,0.00000
8,XpVt6Z1Gjjo,3,0,happy one year vlogaversary,0.80000
9,XpVt6Z1Gjjo,0,0,shit brother may single handedly ruined youtub...,-0.02381


In [ ]:
# 2.Classify the sentiment based on polarity
comments['sentiment_bin'] = comments['polarity'].apply(lambda x: 'positive' if x > 0 else ('negative' if x < 0 else 'neutral'))

In [81]:
comments.head() # Display the first 5 rows of the dataset

,video_id,likes,replies,tokenized_text,polarity,sentiment_bin
0,XpVt6Z1Gjjo,4,0,logan paul yo big day,0.0,neutral
1,XpVt6Z1Gjjo,3,0,following start vine channel seen vlogs,0.0,neutral
2,XpVt6Z1Gjjo,3,0,say hi kong maverick,0.0,neutral
3,XpVt6Z1Gjjo,3,0,fan attendance,0.0,neutral
4,XpVt6Z1Gjjo,3,0,trending,0.0,neutral


In [95]:
sentiment_df = comments.groupby('video_id')['sentiment_bin'].value_counts().unstack().fillna(0)# Count the number of unique values in the 'sentiment_bin' column

In [96]:
sentiment_df # Display the first 10 rows of the sentiment DataFrame

sentiment_bin,negative,neutral,positive
video_id,,,
--JinobXWPk,18.0,52.0,30.0
-1fzGnFwz9M,19.0,28.0,53.0
-3AGlBYyLjo,2.0,2.0,0.0
-5sCWsLlTCI,17.0,22.0,28.0
-6Zc8Co2H3w,34.0,173.0,193.0
...,...,...,...
zqE-ultsWt0,94.0,195.0,211.0
zrOHeEA14kQ,114.0,249.0,134.0
zuKX0fPlo2Q,0.0,0.0,2.0


In [99]:
# 3.Calculate Positive-to-Negative Ratio
sentiment_df['p_n_comment_ratio'] = sentiment_df['positive'] / sentiment_df['negative']
# Replace inf and -inf with NaN
sentiment_df['p_n_comment_ratio'].replace([np.inf, -np.inf], np.nan, inplace=True)
sentiment_df['p_n_comment_ratio'].fillna(0, inplace=True) # Handle cases where there are no negative comments


C:\Users\New Owner\AppData\Local\Temp\ipykernel_19268\1589972635.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  sentiment_df['p_n_comment_ratio'].replace([np.inf, -np.inf], np.nan, inplace=True)
C:\Users\New Owner\AppData\Local\Temp\ipykernel_19268\1589972635.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting

In [104]:
# 4.Calculate Percentage of Neutral Comments used in the analysis
# neutral percentage is used to help calculate the emoti0onal charge related to audience reaction.
sentiment_df['neutral_percentage'] = (sentiment_df['neutral'] / (sentiment_df['positive'] + sentiment_df['negative'] + sentiment_df['neutral'])) * 100
sentiment_df['neutral_percentage'] = sentiment_df['neutral_percentage'].round(2) # Round the percentage to 2 decimal places

In [105]:
sentiment_df

sentiment_bin,negative,neutral,positive,p_n_comment_ratio,neutral_percentage
video_id,,,,,
--JinobXWPk,18.0,52.0,30.0,1.666667,52.00
-1fzGnFwz9M,19.0,28.0,53.0,2.789474,28.00
-3AGlBYyLjo,2.0,2.0,0.0,0.000000,50.00
-5sCWsLlTCI,17.0,22.0,28.0,1.647059,32.84
-6Zc8Co2H3w,34.0,173.0,193.0,5.676471,43.25
...,...,...,...,...,...
zqE-ultsWt0,94.0,195.0,211.0,2.244681,39.00
zrOHeEA14kQ,114.0,249.0,134.0,1.175439,50.10
zuKX0fPlo2Q,0.0,0.0,2.0,0.000000,0.00


In [113]:
sentiment_df.merge(videos[['video_id', 'title', 'l_d_ratio']], on='video_id') # Merge the sentiment DataFrame with the videos DataFrame

,video_id,negative,neutral,positive,p_n_comment_ratio,neutral_percentage,title,l_d_ratio
0,8wNr-NQImFg,98.0,139.0,163.0,1.663265,34.75,"The Check In: HUD, Ben Carson and Hurricanes",33.619529
1,WYYvHb03Eog,137.0,388.0,275.0,2.007299,48.50,Apple iPhone X first look,5.498679
2,XpVt6Z1Gjjo,151.0,411.0,238.0,1.576159,51.38,1 YEAR OF VLOGGING -- HOW LOGAN PAUL CHANGED Y...,53.962738
3,_ANP3HR1jsM,64.0,247.0,187.0,2.921875,49.60,ATTACKED BY A POLICE DOG!!,102.783186
4,_HTXMhKWqnA,76.0,163.0,161.0,2.118421,40.75,iPhone X Impressions & Hands On!,33.973394
5,cLdxuaxaQwc,245.0,286.0,268.0,1.093878,35.79,My Response,14.496832
6,cMKX2tE5Luk,75.0,339.0,286.0,3.813333,48.43,The Disaster Artist | Official Trailer HD | A24,63.431985
7,sjlHnJvXdQs,130.0,382.0,288.0,2.215385,47.75,iPhone X (parody),170.186620
8,zgLtEob6X-Q,70.0,402.0,228.0,3.257143,57.43,Honest Trailers - The Mummy (2017),34.103645


In [114]:
videos

,video_id,title,channel_title,category_id,tags,views,likes,dislikes,comment_total,thumbnail_link,l_d_ratio
0,XpVt6Z1Gjjo,1 YEAR OF VLOGGING -- HOW LOGAN PAUL CHANGED Y...,Logan Paul Vlogs,24,logan paul vlog|logan paul|logan|paul|olympics...,4394029,320053,5931,46245,https://i.ytimg.com/vi/XpVt6Z1Gjjo/default.jpg,53.962738
1,K4wEI5zhHB0,iPhone X — Introducing iPhone X — Apple,Apple,28,Apple|iPhone 10|iPhone Ten|iPhone|Portrait Lig...,7860119,185853,26679,0,https://i.ytimg.com/vi/K4wEI5zhHB0/default.jpg,6.966266
2,cLdxuaxaQwc,My Response,PewDiePie,22,[none],5845909,576597,39774,170708,https://i.ytimg.com/vi/cLdxuaxaQwc/default.jpg,14.496832
3,WYYvHb03Eog,Apple iPhone X first look,The Verge,28,apple iphone x hands on|Apple iPhone X|iPhone ...,2642103,24975,4542,12829,https://i.ytimg.com/vi/WYYvHb03Eog/default.jpg,5.498679
4,sjlHnJvXdQs,iPhone X (parody),jacksfilms,23,jacksfilms|parody|parodies|iphone|iphone x|iph...,1168130,96666,568,6666,https://i.ytimg.com/vi/sjlHnJvXdQs/default.jpg,170.186620
5,cMKX2tE5Luk,The Disaster Artist | Official Trailer HD | A24,A24,1,a24|a24 films|a24 trailers|independent films|t...,1311445,34507,544,3040,https://i.ytimg.com/vi/cMKX2tE5Luk/default.jpg,63.431985
6,8wNr-NQImFg,"The Check In: HUD, Ben Carson and Hurricanes",Late Night with Seth Meyers,23,Late night|Seth Meyers|check in|hud|Ben Carson...,666169,9985,297,1071,https://i.ytimg.com/vi/8wNr-NQImFg/default.jpg,33.619529
7,_HTXMhKWqnA,iPhone X Impressions & Hands On!,Marques Brownlee,28,iPhone X|iphone x|iphone 10|iPhone X impressio...,1728614,74062,2180,15297,https://i.ytimg.com/vi/_HTXMhKWqnA/default.jpg,33.973394
8,_ANP3HR1jsM,ATTACKED BY A POLICE DOG!!,RomanAtwoodVlogs,22,Roman Atwood|Roman|Atwood|roman atwood vlogs|f...,1338533,69687,678,5643,https://i.ytimg.com/vi/_ANP3HR1jsM/default.jpg,102.783186
9,zgLtEob6X-Q,Honest Trailers - The Mummy (2017),Screen Junkies,1,screenjunkies|screen junkies|screenjunkies new...,1056891,29943,878,4046,https://i.ytimg.com/vi/zgLtEob6X-Q/default.jpg,34.103645


In [ ]:
videos.info()

In [ ]:
#videos_mass = videos.merge(sent_prop[['comment_sent_ratio']], left_on='video_id', right_index=True, how='left') # Merge the sentiment ratio with the videos DataFrame

In [ ]:
sent_prop.head(10)